In [3]:
import pandas as pd

path = "../../data/ws/full/station_id=1/year=2024/data.parquet"
# 1 ate 83
# 2012 ate 2025

df = pd.read_parquet(path)  

print(df.columns)
print(" ")
print(df.head())

Index(['id', 'nome', 'type', 'latitude', 'longitude', 'rains', 'm5', 'm15',
       'h01', 'h04', 'h24', 'h96', 'mes', 'observation_datetime',
       'request_datetime', 'year'],
      dtype='object')
 
   id     nome type  latitude  longitude  rains   m5  m15  h01  h04  h24  h96  \
0   1  Adeus 1  plv  -22.8636   -43.2636    0.0  0.0  0.0  0.0  0.0  0.8  8.6   
1   1  Adeus 1  plv  -22.8636   -43.2636    0.0  0.0  0.0  0.0  0.0  0.8  8.6   
2   1  Adeus 1  plv  -22.8636   -43.2636    0.0  0.0  0.0  0.0  0.0  0.8  8.6   
3   1  Adeus 1  plv  -22.8636   -43.2636    0.0  0.0  0.0  0.0  0.0  0.8  8.6   
4   1  Adeus 1  plv  -22.8636   -43.2636    0.0  0.0  0.0  0.0  0.0  0.8  8.6   

    mes      observation_datetime          request_datetime  year  
0  59.8 2024-01-01 00:00:00+00:00 2024-01-01 00:00:00+00:00  2024  
1  59.8 2024-01-01 00:15:00+00:00 2024-01-01 00:20:00+00:00  2024  
2  59.8 2024-01-01 00:30:00+00:00 2024-01-01 00:30:00+00:00  2024  
3  59.8 2024-01-01 00:45:00+00:00 2024-

## grid do radar com os pixels

In [5]:
import numpy as np

grid = np.load("../src/datasets/sumare_radar_latlon_grid.npz")

lat_grid = grid["lat"]
lon_grid = grid["lon"]

print(lat_grid.shape)
print(lon_grid.shape)

(654, 656)
(654, 656)


## carrega todas as estaçoes web sirene

In [6]:
import pandas as pd

stations = []

for station_id in range(1, 84):

    path = f"../../data/ws/full/station_id={station_id}/year=2024/data.parquet"

    try:
        df = pd.read_parquet(path)

        # pega só uma linha da estação
        station_info = df[
            ["id", "nome", "latitude", "longitude"]
        ].drop_duplicates()

        stations.append(station_info)

        print(f"OK estação {station_id}")

    except Exception as e:
        print(f"Erro estação {station_id}: {e}")

# junta tudo
stations_df = pd.concat(stations, ignore_index=True)

# remove duplicatas finais
stations_df = stations_df.drop_duplicates()

print(stations_df.head())

print(" ")
print("Quantidade de estações:", len(stations_df))

OK estação 1
OK estação 2
OK estação 3
OK estação 4
OK estação 5
OK estação 6
OK estação 7
OK estação 8
OK estação 9
OK estação 10
OK estação 11
OK estação 12
OK estação 13
OK estação 14
OK estação 15
OK estação 16
OK estação 17
OK estação 18
OK estação 19
OK estação 20
OK estação 21
OK estação 22
OK estação 23
OK estação 24
OK estação 25
OK estação 26
OK estação 27
OK estação 28
OK estação 29
OK estação 30
OK estação 31
OK estação 32
OK estação 33
OK estação 34
OK estação 35
OK estação 36
OK estação 37
OK estação 38
OK estação 39
OK estação 40
OK estação 41
OK estação 42
OK estação 43
OK estação 44
OK estação 45
OK estação 46
OK estação 47
OK estação 48
OK estação 49
OK estação 50
OK estação 51
OK estação 52
OK estação 53
OK estação 54
OK estação 55
OK estação 56
OK estação 57
OK estação 58
OK estação 59
OK estação 60
OK estação 61
OK estação 62
OK estação 63
OK estação 64
OK estação 65
OK estação 66
OK estação 67
OK estação 68
OK estação 69
OK estação 70
OK estação 71
OK estação 72
O

In [7]:
print(stations_df.head())
print(" ")
print(len(stations_df))

   id        nome  latitude  longitude
0   1     Adeus 1  -22.8636   -43.2636
1   2   AlemÃ£o 1  -22.8547   -43.2725
2   3  AndaraÃ­ 1  -22.9327   -43.2567
3   4    Baiana 1  -22.8595   -43.2657
4   5    BarÃ£o 1  -22.9023   -43.3434
 
83


## mapeia as estaçoes no pixel mais proximo do radar

In [8]:
mapping = []

for _, row in stations_df.iterrows():

    station_id = row["id"]
    station_name = row["nome"]

    station_lat = row["latitude"]
    station_lon = row["longitude"]

    # distância entre estação e TODOS os pixels
    dist = (
        (lat_grid - station_lat) ** 2 +
        (lon_grid - station_lon) ** 2
    )

    # pixel mais próximo
    pixel_i, pixel_j = np.unravel_index(
        np.argmin(dist),
        dist.shape
    )

    mapping.append({
        "station_id": station_id,
        "nome": station_name,
        "latitude": station_lat,
        "longitude": station_lon,
        "pixel_i": pixel_i,
        "pixel_j": pixel_j,
    })

# transforma em dataframe
mapping_df = pd.DataFrame(mapping)

In [9]:
print(mapping_df.head())

print(" ")
print("Quantidade de estações mapeadas:", len(mapping_df))

   station_id        nome  latitude  longitude  pixel_i  pixel_j
0           1     Adeus 1  -22.8636   -43.2636      303      324
1           2   AlemÃ£o 1  -22.8547   -43.2725      300      322
2           3  AndaraÃ­ 1  -22.9327   -43.2567      321      325
3           4    Baiana 1  -22.8595   -43.2657      302      323
4           5    BarÃ£o 1  -22.9023   -43.3434      313      304
 
Quantidade de estações mapeadas: 83


In [11]:
mapping_df.to_csv(
    "../src/datasets/mapeamento_pixel_estacao.csv",
    index=False
)